# Dataset Inventory Dashboard

This notebook reads the CSV outputs generated by `scripts/report_dataset_info.py` and creates project-management summary tables and plots for cohort balance, session/run coverage, and missing-data review.


## How to Use

Run `scripts/report_dataset_info.py BIDS_DIR --csv OUTPUT_CSV` first. Then set `REPORT_CSV` below, or set the `DATASET_INFO_CSV` environment variable before launching Jupyter.


In [ ]:
from pathlib import Path
import os

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

plt.style.use('seaborn-v0_8-whitegrid')
pd.set_option('display.max_rows', 200)
pd.set_option('display.max_columns', 100)
pd.set_option('display.width', 160)

REPORT_CSV = Path(os.environ.get('DATASET_INFO_CSV', '../derivatives/qc/provenance/dataset_info.csv')).expanduser()
if not REPORT_CSV.exists() and Path('dataset_info.csv').exists():
    REPORT_CSV = Path('dataset_info.csv')

if not REPORT_CSV.exists():
    raise FileNotFoundError(f'Session CSV not found: {REPORT_CSV}. Update REPORT_CSV in this cell or launch Jupyter with DATASET_INFO_CSV=/path/to/dataset_info.csv.')

FIGURE_DIR = REPORT_CSV.parent / 'dataset_info_figures'
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

print(f'Reading session-level report: {REPORT_CSV}')
print(f'Figures will be saved to: {FIGURE_DIR}')


In [ ]:
def companion_csv(path, suffix):
    return path.with_name(f'{path.stem}_{suffix}{path.suffix}')

SESSION_CSV = REPORT_CSV
GROUP_CSV = companion_csv(REPORT_CSV, 'group_summary')
SUBJECT_CSV = companion_csv(REPORT_CSV, 'subject_summary')
RUN_CSV = companion_csv(REPORT_CSV, 'run_inventory')

required_columns = {'subject', 'group', 'session', 'available_folders', 'anat_count', 'func_run_count', 'fmap_count', 'dwi_count', 'run_labels'}
sessions = pd.read_csv(SESSION_CSV).fillna('')
missing_columns = required_columns - set(sessions.columns)
if missing_columns:
    raise ValueError(f'Session CSV is missing expected columns: {sorted(missing_columns)}')

for column in ['anat_count', 'func_run_count', 'fmap_count', 'dwi_count']:
    sessions[column] = pd.to_numeric(sessions[column], errors='coerce').fillna(0).astype(int)

sessions['folder_set'] = sessions['available_folders'].apply(lambda value: set(str(value).split(';')) if str(value) else set())
sessions['has_anat_folder'] = sessions['folder_set'].apply(lambda folders: 'anat' in folders)
sessions['has_func_folder'] = sessions['folder_set'].apply(lambda folders: 'func' in folders)
sessions['has_fmap_folder'] = sessions['folder_set'].apply(lambda folders: 'fmap' in folders)
sessions['has_anat_file'] = sessions['anat_count'] > 0
sessions['has_func_run'] = sessions['func_run_count'] > 0
sessions['has_fmap_file'] = sessions['fmap_count'] > 0
sessions['session_complete'] = sessions['has_anat_folder'] & sessions['has_func_folder'] & sessions['has_func_run'] & sessions['has_fmap_folder']

def missing_reasons(row):
    reasons = []
    if not row['has_anat_folder']:
        reasons.append('missing anat folder')
    elif not row['has_anat_file']:
        reasons.append('anat folder has no image')
    if not row['has_func_folder']:
        reasons.append('missing func folder')
    elif not row['has_func_run']:
        reasons.append('no BOLD runs')
    if not row['has_fmap_folder']:
        reasons.append('missing fmap folder')
    elif not row['has_fmap_file']:
        reasons.append('fmap folder has no files')
    return '; '.join(reasons)

sessions['missing_reasons'] = sessions.apply(missing_reasons, axis=1)

subject_status = sessions.groupby(['subject', 'group'], as_index=False).agg(
    session_count=('session', 'nunique'),
    total_func_runs=('func_run_count', 'sum'),
    complete_sessions=('session_complete', 'sum'),
    incomplete_sessions=('session_complete', lambda values: int((~values).sum())),
)
subject_status['subject_complete'] = subject_status['incomplete_sessions'] == 0
subject_missing = sessions.loc[~sessions['session_complete'], ['subject', 'group', 'session', 'missing_reasons']].copy()

if GROUP_CSV.exists():
    groups = pd.read_csv(GROUP_CSV).fillna('')
else:
    groups = subject_status.groupby('group', as_index=False).agg(subject_count=('subject', 'nunique'))

if SUBJECT_CSV.exists():
    subjects = pd.read_csv(SUBJECT_CSV).fillna('')
else:
    subjects = subject_status.copy()

if RUN_CSV.exists():
    runs = pd.read_csv(RUN_CSV).fillna('')
else:
    run_rows = []
    for _, row in sessions.iterrows():
        labels = [label for label in str(row['run_labels']).split(';') if label]
        for label in labels:
            run_rows.append({'subject': row['subject'], 'group': row['group'], 'session': row['session'], 'label': label})
    runs = pd.DataFrame(run_rows)

print(f'Sessions loaded: {len(sessions)}')
print(f'Subjects loaded: {subject_status["subject"].nunique()}')
print(f'Runs loaded: {len(runs)}')


## Executive Summary Tables


In [ ]:
summary = pd.DataFrame([
    {'metric': 'Total subjects', 'value': subject_status['subject'].nunique()},
    {'metric': 'Complete subjects', 'value': int(subject_status['subject_complete'].sum())},
    {'metric': 'Subjects with missing data', 'value': int((~subject_status['subject_complete']).sum())},
    {'metric': 'Total sessions', 'value': len(sessions)},
    {'metric': 'Complete sessions', 'value': int(sessions['session_complete'].sum())},
    {'metric': 'Sessions with missing data', 'value': int((~sessions['session_complete']).sum())},
    {'metric': 'Total functional runs', 'value': int(sessions['func_run_count'].sum())},
])
display(summary)
display(groups)
display(subject_status.sort_values(['subject_complete', 'group', 'subject']))


## Missing Data Review


In [ ]:
if subject_missing.empty:
    print('No missing data flags found under the current completeness definition.')
else:
    display(subject_missing.sort_values(['group', 'subject', 'session']))

reason_counts = []
for reasons in subject_missing['missing_reasons']:
    for reason in str(reasons).split('; '):
        if reason:
            reason_counts.append(reason)
missing_reason_counts = pd.Series(reason_counts).value_counts().rename_axis('missing_reason').reset_index(name='session_count')
display(missing_reason_counts)


## Inventory Tables


In [ ]:
display(sessions.drop(columns=['folder_set']).sort_values(['group', 'subject', 'session']))
if len(runs):
    display(runs.sort_values([column for column in ['group', 'subject', 'session', 'run', 'label'] if column in runs.columns]))
else:
    print('No run inventory rows found.')


## Cohort And Completeness Plots


In [ ]:
def save_current_figure(name):
    path = FIGURE_DIR / name
    plt.tight_layout()
    plt.savefig(path, dpi=200, bbox_inches='tight')
    print(f'Saved: {path}')

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
subject_status['group'].value_counts().reindex(['patient', 'HC', 'unknown']).dropna().plot(kind='bar', ax=axes[0], color=['#4C78A8', '#F58518', '#7F7F7F'])
axes[0].set_title('Subjects by group')
axes[0].set_xlabel('Group')
axes[0].set_ylabel('Subjects')
subject_status['subject_complete'].map({True: 'complete', False: 'missing data'}).value_counts().reindex(['complete', 'missing data']).fillna(0).plot(kind='bar', ax=axes[1], color=['#54A24B', '#E45756'])
axes[1].set_title('Subject completeness')
axes[1].set_xlabel('Status')
axes[1].set_ylabel('Subjects')
save_current_figure('cohort_subject_completeness.png')
plt.show()


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
if missing_reason_counts.empty:
    axes[0].text(0.5, 0.5, 'No missing data flags', ha='center', va='center')
    axes[0].set_axis_off()
else:
    missing_reason_counts.sort_values('session_count').plot(kind='barh', x='missing_reason', y='session_count', legend=False, ax=axes[0], color='#E45756')
    axes[0].set_title('Missing-data reasons')
    axes[0].set_xlabel('Sessions')
    axes[0].set_ylabel('')
subject_status.sort_values('session_count').plot(kind='barh', x='subject', y='session_count', legend=False, ax=axes[1], color='#4C78A8')
axes[1].set_title('Session count per subject')
axes[1].set_xlabel('Sessions')
axes[1].set_ylabel('Subject')
save_current_figure('missing_reasons_and_session_counts.png')
plt.show()


In [ ]:
plot_data = sessions.sort_values(['group', 'subject', 'session']).copy()
plot_data['subject_session'] = plot_data['subject'] + ' / ' + plot_data['session']
fig_height = max(4, 0.28 * len(plot_data))
fig, ax = plt.subplots(figsize=(10, fig_height))
colors = plot_data['session_complete'].map({True: '#54A24B', False: '#E45756'})
ax.barh(plot_data['subject_session'], plot_data['func_run_count'], color=colors)
ax.set_title('Functional runs per session')
ax.set_xlabel('BOLD runs')
ax.set_ylabel('Subject / session')
for index, row in plot_data.reset_index(drop=True).iterrows():
    label = 'complete' if row['session_complete'] else row['missing_reasons']
    ax.text(row['func_run_count'] + 0.05, index, label, va='center', fontsize=8)
save_current_figure('functional_runs_per_session.png')
plt.show()


In [ ]:
heatmap = sessions.pivot_table(index='subject', columns='session', values='session_complete', aggfunc='max').sort_index()
heatmap_numeric = heatmap.astype(float)
fig_width = max(6, 0.7 * len(heatmap_numeric.columns))
fig_height = max(4, 0.3 * len(heatmap_numeric.index))
fig, ax = plt.subplots(figsize=(fig_width, fig_height))
im = ax.imshow(heatmap_numeric.values, aspect='auto', cmap='RdYlGn', vmin=0, vmax=1)
ax.set_xticks(np.arange(len(heatmap_numeric.columns)))
ax.set_xticklabels(heatmap_numeric.columns, rotation=45, ha='right')
ax.set_yticks(np.arange(len(heatmap_numeric.index)))
ax.set_yticklabels(heatmap_numeric.index)
ax.set_title('Subject by session completeness')
for row_index in range(heatmap_numeric.shape[0]):
    for col_index in range(heatmap_numeric.shape[1]):
        value = heatmap_numeric.iloc[row_index, col_index]
        if not np.isnan(value):
            ax.text(col_index, row_index, 'OK' if value == 1 else 'MISS', ha='center', va='center', fontsize=8, color='black')
cbar = fig.colorbar(im, ax=ax, fraction=0.025, pad=0.02)
cbar.set_ticks([0, 1])
cbar.set_ticklabels(['missing', 'complete'])
save_current_figure('subject_session_completeness_heatmap.png')
plt.show()


## Export Review Tables


In [ ]:
subject_status_path = FIGURE_DIR / 'subject_completeness_table.csv'
missing_path = FIGURE_DIR / 'missing_data_review_table.csv'
session_review_path = FIGURE_DIR / 'session_review_table.csv'
subject_status.to_csv(subject_status_path, index=False)
subject_missing.to_csv(missing_path, index=False)
sessions.drop(columns=['folder_set']).to_csv(session_review_path, index=False)
print(f'Wrote: {subject_status_path}')
print(f'Wrote: {missing_path}')
print(f'Wrote: {session_review_path}')
